In [42]:
!pip install torchmetrics

In [43]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import re
from tabulate import tabulate
from tqdm import trange
import random
from torchmetrics.classification import Recall, Accuracy, AUROC, Precision
from functools import partial

In [44]:
# --- DATA LOADING ---

# Load the local NEISS CSV file
# We force dtype=str to preserve codes (e.g., '055') and handle mixed types
file_path = 'PoisonedOnly_NEISS_2004-2023.xlsx'
print(f"Loading data from {file_path}...")

df = pd.read_excel(file_path, sheet_name='ALL (DO NOT EDIT)', dtype=str)

# Fill NaN values in Narrative to avoid errors during processing
if 'Narrative' in df.columns:
    df['Narrative'] = df['Narrative'].fillna('')

print(f"Total rows loaded: {len(df)}")
df.head()

Loading data from PoisonedOnly_NEISS_2004-2023.xlsx...
Total rows loaded: 91355


,CPSC_Case_Number,Treatment_Date,Age,Sex,Race,Other_Race,Hispanic,Body_Part,Diagnosis,Other_Diagnosis,...,Fire_Involvement,Product_1,Product_2,Product_3,Alcohol,Drug,Narrative,Stratum,PSU,Weight
0,40101637,2004-01-04 00:00:00,25,2,0,NaN,NaN,85,68,NaN,...,0,983,0,0,NaN,NaN,"PT W/HIVES ON TORSO SINCE THIS AM. SWELLING TO FINGERS, BLURRED VISION.RECENT URI. BEGAN NEW MED *** 1 MO AGO. USED NEW SOAP. D:URTICARIA.",L,89,58.9999
1,40101649,2004-01-02 00:00:00,2,1,0,NaN,NaN,85,68,NaN,...,0,1930,1135,0,NaN,NaN,CHILD INGESTED ABOUT 1/3 TO 1/2 TO 120 ML BOTTLE OF *** ELIXER. NO SYMPTOMS. OCCURRED 1.5 HOURS AGO. D: *** INGESTION.,L,89,58.9999
2,40102391,2004-01-02 00:00:00,2,1,0,NaN,NaN,85,68,NaN,...,0,1931,0,0,NaN,NaN,Format name,S,48,69.894
3,40103631,2004-01-02 00:00:00,2,1,1,NaN,NaN,85,68,NaN,...,0,1930,0,0,NaN,NaN,ACCIDENTAL *** INGESTION-@ HOME-CLIMBED UP & TOOK 78 ML OF CHILDREN'S *** @ 160 MG PER ML,V,53,16.7808
4,40105546,2004-01-02 00:00:00,34,1,1,NaN,NaN,85,68,NaN,...,0,1807,0,0,NaN,NaN,FOUND ON FLOOR SEIZING AT JAIL-TAKES *** FOR SEIZURES-C/O HEADACHESEIZURE DUE TO SUBTHERAPEUTIC LEVEL OF ***,S,92,69.894


In [45]:
# --- WEAK SUPERVISION LOGIC (UPDATED) ---

def get_weak_label(row):
    """
    Assigns a training label based on Product Codes and Narrative keywords.

    Labels:
      1:  Positive Class (Vitamins, Supplements, Herbs, Homeopathic)
      0:  Negative Class (Pharmaceuticals, Iron, Chemicals, Household items, Alcohol)
     -1:  Ambiguous (Discarded from training, used for Inference later)
    """

    # Convert inputs to string and uppercase for case-insensitive matching
    prod_code = str(row['Product_1'])
    text = str(row['Narrative']).upper()

    # --- RULE 1: EXCLUSIONS (Label 0 - NON-VITAMINS) ---
    # We must rigorously define what is NOT a vitamin to prevent Model Collapse.

    # 1.1: Specific Product Codes
    # Code 1916 is IRON (Clinically dangerous, distinct from standard vitamins)
    if prod_code == '1916':
        return 0

    # 1.2: Negative Keywords (Expanded List)
    # If the narrative contains ANY of these, it is likely a drug or toxic substance.
    danger_keywords = [
        # --- Pharmaceuticals ---
        "IRON", "FEOSOL", "FERROUS",
        "TYLENOL", "ACETAMINOPHEN", "IBUPROFEN", "MOTRIN", "ADVIL",
        "ASPIRIN", "BUPROPION", "MEDICATION", "DRUG", "PRESCRIPTION",
        "ANTIBIOTIC", "AMOXICILLIN", "PILL", "TABLET", "CAPSULE", # Generic terms often imply meds unless "Vitamin" is specified

        # --- Household Chemicals & Adhesives (The previous failure points) ---
        "BLEACH", "CLEANER", "DETERGENT", "SOAP", "SHAMPOO", "LOTION", "POD",
        "GLUE", "ADHESIVE", "PASTE", "SUPERGLUE", "SHOE GOO", "CEMENT",
        "PAINT", "POLISH", "REMOVER", "ACETONE", "SOLVENT", "GAS", "KEROSENE",

        # --- Alcohol ---
        "ALCOHOL", "ETHANOL", "VODKA", "BEER", "WINE", "LIQUOR", "WHISKEY",
        "RUBBING", "SANITIZER",

        # --- Miscellaneous Dangerous Items ---
        "BATTERY", "MAGNET", "TOY", "BEAD", "CIGARETTE", "TOBACCO", "VAPE"
    ]

    if any(keyword in text for keyword in danger_keywords):
        return 0

    # --- RULE 2: INCLUSIONS (Label 1 - VITAMINS) ---

    target_keywords = [
        "VITAMIN", "VIT ", "MULTIVITAMIN", "MULTI-VITAMIN",
        "MELATONIN", "GUMMY", "GUMMIES",
        "FISH OIL", "OMEGA", "PROBIOTIC",
        "ASHWAGANDHA", "TURMERIC", "MAGNESIUM", "ZINC", "CALCIUM",
        "SUPPLEMENT", "HERBAL", "HOMEOPATHIC", "ELECTROLYTE"
    ]

    # Check 1: Explicit Vitamin Product Codes
    # 1927 = Liquid Vitamins, 1931 = Tablet Vitamins
    if prod_code in ['1927', '1931']:
        return 1

    # Check 2: "Other" Code (1932) BUT contains a target keyword
    if prod_code == '1932':
        if any(keyword in text for keyword in target_keywords):
            return 1

    # Check 3: Narrative explicitly mentions a strong target keyword
    # (Overrides generic codes if no danger keywords were found above)
    if any(keyword in text for keyword in target_keywords):
        return 1

    # --- RULE 3: AMBIGUOUS (Label -1) ---
    # Cases with redacted text (***) or generic codes without clear keywords.
    # These will be labeled by the BERT model during the Inference phase.
    return -1

# Apply the function to the dataset
print("Applying updated Weak Supervision rules...")
df['label'] = df.apply(get_weak_label, axis=1)

# Check the new distribution
print("\nLabel Distribution (Training Set vs Ambiguous):")
print(df['label'].value_counts())

Applying updated Weak Supervision rules...

Label Distribution (Training Set vs Ambiguous):
label
 0    49133
-1    32818
 1     9404
Name: count, dtype: int64


In [46]:
# --- FILTERING & PREPARATION ---

# Keep only clear cases (0 and 1). Discard ambiguous (-1).
df_clean = df[df['label'] != -1].copy()

# Rename 'Narrative' to 'text' to match the BERT pipeline expectations
df_clean = df_clean.rename(columns={'Narrative': 'text'})

# Reset index after filtering
df_clean = df_clean.reset_index(drop=True)

print(f"\nFinal Training Set Size: {len(df_clean)}")
print("Class Balance:")
print(df_clean['label'].value_counts())

# Extract arrays for the model
text = df_clean.text.values
labels = df_clean.label.values.astype(int) # Ensure labels are integers

# Display a few examples to verify
print("\nSample Data:")
print(df_clean[['Product_1', 'text', 'label']].head(10))


Final Training Set Size: 58537
Class Balance:
label
0    49133
1     9404
Name: count, dtype: int64

Sample Data:
  Product_1  \
0       983   
1      1931   
2      4074   
3      1135   
4      1913   
5       909   
6      1817   
7      1927   
8      1931   
9       956   

                                                                                                                                            text  \
0     PT W/HIVES ON TORSO SINCE THIS AM. SWELLING TO FINGERS, BLURRED VISION.RECENT URI. BEGAN NEW MED *** 1 MO AGO. USED NEW SOAP. D:URTICARIA.   
1                                                                                                                                    Format name   
2                                                                            PT FOUND SLUMPED IN CHAIR, PALE, SOB, DX ACUTE ALCOHOL INTOXICATION   
3     MEDICATION OVERDOSE - 52 YOM PRESENTS WEAK AND DIFFICULTY WITH AMBULATION, MIXES UP HIS MEDICATIONS AND THERE ARE ABOUT 5

In [47]:
# --- TOKENIZATION & SPLIT ---

# Initialize Tokenizer (do_lower_case=True is important for uppercase NEISS text)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# Helper function to tokenize properly
# FIX: Use tokenizer(...) directly instead of tokenizer.encode_plus
def preprocessing(input_text, tokenizer):
    return tokenizer(
        str(input_text), # Ensure input is a string
        add_special_tokens=True,
        max_length=32,  # Short narratives, 32 is usually enough
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

# Prepare lists for tensors
token_id = []
attention_masks = []

print("Tokenizing data...")
# Wrap in try-except to debug specific bad rows if needed
try:
    for sample in text:
        encoding_dict = preprocessing(sample, tokenizer)
        # We need to flatten the extra dimension added by return_tensors='pt'
        token_id.append(encoding_dict['input_ids'])
        attention_masks.append(encoding_dict['attention_mask'])
except Exception as e:
    print(f"Error tokenizing sample: {sample}")
    raise e

# Convert to Tensors
# Concatenate the list of tensors [1, 32] into a single tensor [N, 32]
token_id = torch.cat(token_id, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels_tensor = torch.tensor(labels)

# Train/Validation Split
val_ratio = 0.2
batch_size = 32  # Good standard for BERT

# Ensure labels are numpy array for stratify
if isinstance(labels, list):
    labels = np.array(labels)

train_idx, val_idx = train_test_split(
    np.arange(len(labels)),
    test_size=val_ratio,
    stratify=labels, # Keeps the same % of vitamins in train and val
    random_state=42
)

# Create TensorDatasets
train_set = TensorDataset(token_id[train_idx], attention_masks[train_idx], labels_tensor[train_idx])
val_set = TensorDataset(token_id[val_idx], attention_masks[val_idx], labels_tensor[val_idx])

# Create DataLoaders
train_dataloader = DataLoader(train_set, sampler=RandomSampler(train_set), batch_size=batch_size)
validation_dataloader = DataLoader(val_set, sampler=SequentialSampler(val_set), batch_size=batch_size)

print("Data loaded and tokenized successfully!")

Tokenizing data...
Data loaded and tokenized successfully!


In [48]:
# --- LORA MODEL DEFINITION ---

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = torch.nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

# Load Pre-trained BERT
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2, # 0 = Other/Danger, 1 = Vitamin
    output_attentions=False,
    output_hidden_states=False
)

# Freeze all parameters first
for param in model.parameters():
    param.requires_grad = False

# Inject LoRA layers into Query and Value projections
lora_rank = 8
lora_alpha = 16

assign_lora = partial(LinearWithLoRA, rank=lora_rank, alpha=lora_alpha)

for layer in model.bert.encoder.layer:
    layer.attention.self.query = assign_lora(layer.attention.self.query)
    layer.attention.self.value = assign_lora(layer.attention.self.value)

# Unfreeze the Classifier Head (we need to retrain the final decision layer)
for param in model.classifier.parameters():
    param.requires_grad = True

# Check trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters (LoRA + Head): {trainable_params:,}")
print(f"Percentage Trainable: {100 * trainable_params / total_params:.2f}%")

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total Parameters: 109,778,690
Trainable Parameters (LoRA + Head): 296,450
Percentage Trainable: 0.27%


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): LinearWithLoRA(
                (linear): Linear(in_features=768, out_features=768, bias=True)
                (lora): LoRALayer()
              )
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): LinearWithLoRA(
                (linear): Linear(in_features=768, out_features=768, bias=True)
                (lora): LoRALayer()
              )
              (dropout): Dropout(p=0.1, inplace=Fa

In [49]:
# --- TRAINING LOOP ---

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5) # Higher LR for LoRA is common

# Metrics
accuracy = Accuracy(task="binary").to(device)
precision = Precision(task="binary").to(device)
recall = Recall(task="binary").to(device)
#f1 = F1Score(task="binary").to(device) # Se non hai F1Score importato, usa precision/recall

epochs = 5

print("Starting training...")

for epoch in trange(epochs, desc='Epoch'):

    # === TRAIN ===
    model.train()
    tr_loss = 0
    nb_tr_steps = 0

    for batch in train_dataloader:
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_attn_masks, b_labels = batch

        optimizer.zero_grad()

        outputs = model(b_input_ids, attention_mask=b_attn_masks, labels=b_labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        tr_loss += loss.item()
        nb_tr_steps += 1

    # === VALIDATION ===
    model.eval()
    val_accuracy = []
    val_precision = []
    val_recall = []

    for batch in validation_dataloader:
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_attn_masks, b_labels = batch

        with torch.no_grad():
            outputs = model(b_input_ids, attention_mask=b_attn_masks)

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        # Calculate metrics
        val_accuracy.append(accuracy(preds, b_labels).item())
        val_precision.append(precision(preds, b_labels).item())
        val_recall.append(recall(preds, b_labels).item())

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {tr_loss/nb_tr_steps:.4f}")
    print(f"Val Accuracy:  {np.mean(val_accuracy):.4f}")
    print(f"Val Precision: {np.mean(val_precision):.4f}")
    print(f"Val Recall:    {np.mean(val_recall):.4f}")

Starting training...


Epoch:  20%|██        | 1/5 [00:53<03:34, 53.53s/it]


Epoch 1
Train Loss: 0.1729
Val Accuracy:  0.9520
Val Precision: 0.8731
Val Recall:    0.8183


Epoch:  40%|████      | 2/5 [01:45<02:38, 52.75s/it]


Epoch 2
Train Loss: 0.1246
Val Accuracy:  0.9628
Val Precision: 0.8770
Val Recall:    0.8983


...


Epoch 3
Train Loss: 0.1143
Val Accuracy:  0.9601
Val Precision: 0.8518
Val Recall:    0.9164


Epoch:  80%|████████  | 4/5 [03:29<00:52, 52.11s/it]


Epoch 4
Train Loss: 0.1195
Val Accuracy:  0.9616
Val Precision: 0.8871
Val Recall:    0.8728


Epoch: 100%|██████████| 5/5 [04:21<00:00, 52.34s/it]


Epoch 5
Train Loss: 0.1084
Val Accuracy:  0.9568
Val Precision: 0.9219
Val Recall:    0.7912


In [50]:
# --- INFERENCE ON AMBIGUOUS DATA (-1) ---

# 1. Select only the ambiguous rows
df_pred = df[df['label'] == -1].copy()
print(f"Running inference on {len(df_pred)} ambiguous cases...")

# 2. Tokenize (Reusable function)
pred_token_id = []
pred_attention_masks = []

# Use a try-except block just in case of weird text
for sample in df_pred['Narrative'].astype(str):
    encoding_dict = preprocessing(sample, tokenizer)
    pred_token_id.append(encoding_dict['input_ids'])
    pred_attention_masks.append(encoding_dict['attention_mask'])

pred_token_id = torch.cat(pred_token_id, dim=0)
pred_attention_masks = torch.cat(pred_attention_masks, dim=0)

# 3. Create DataLoader (No labels needed here)
pred_dataset = TensorDataset(pred_token_id, pred_attention_masks)
pred_dataloader = DataLoader(pred_dataset, sampler=SequentialSampler(pred_dataset), batch_size=32)

# 4. Prediction Loop
print("Predicting...")
model.eval()
predictions = []
probabilities = []

for batch in pred_dataloader:
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_attn_masks = batch

    with torch.no_grad():
        outputs = model(b_input_ids, attention_mask=b_attn_masks)

    logits = outputs.logits

    # Get the class (0 or 1)
    preds = torch.argmax(logits, dim=1).cpu().numpy()
    # Get the probability (confidence)
    probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()[:, 1] # Prob of class 1

    predictions.extend(preds)
    probabilities.extend(probs)

# 5. Save results back to DataFrame
df_pred['predicted_label'] = predictions
df_pred['confidence'] = probabilities

print("Inference done!")

# 6. Show some examples of what the model "discovered"
print("\nExamples of RECOVERED Vitamins (Model said YES but Rules said UNSURE):")
print(df_pred[df_pred['predicted_label'] == 1][['Narrative', 'confidence']].head(10))

print("\nExamples of REJECTED Cases (Model said NO):")
print(df_pred[df_pred['predicted_label'] == 0][['Narrative', 'confidence']].head(10))

Running inference on 32818 ambiguous cases...
Predicting...
Inference done!

Examples of RECOVERED Vitamins (Model said YES but Rules said UNSURE):
                                                                                                                  Narrative  \
3                                 ACCIDENTAL *** INGESTION-@ HOME-CLIMBED UP & TOOK 78 ML OF CHILDREN'S *** @ 160 MG PER ML   
13                     3 YO BF - INGETION - MAY HAVE INGESTED 20 CC'S OF ATUSS WITH CODEINE PRIOR TO ARRI VAL - NO SYMPTOMS   
17                                     CHILD *** INGESTION- DRANK FROM BOTTLE FOUND WITH LIQUID ON CLOTHESDX *** INGESTION.   
22                                          PT DX INGESTION OF *** - MOM STATES SHE TOOK 100MG *** TAB BYMISTAKE AT4:00 PM.   
25                                                                               HIT CAN WITH AXE DX. POLYETHELENE EXPOSURE   
62                       CHILD POSSIBLY INGESTED QUININE SULFATE. UNKNOWN AMOUNT. NO ACUTE

In [51]:
df_pred['predicted_label'].value_counts()

,count
predicted_label,
0,24591
1,8227


In [52]:
# --- BLOCK 8: RESULTS ANALYSIS & SANITY CHECK ---

# Filter specifically for cases predicted as VITAMINS (Label 1)
vitamins_found = df_pred[df_pred['predicted_label'] == 1]
total_vitamins = len(vitamins_found)

print(f"Total 'Recovered' Vitamins from ambiguous data: {total_vitamins}")

# --- CHECK A: CONFIRMATION OF KNOWN CODES ---
# How many of these actually had the correct Vitamin codes (1931/1927)
# but were redacted (***) or missed by strict keywords?
known_code_mask = vitamins_found['Product_1'].isin(['1931', '1927'])
code_confirmed_count = vitamins_found[known_code_mask].shape[0]

print(f"\n1. Code Confirmation:")
print(f"   - {code_confirmed_count} cases ({code_confirmed_count/total_vitamins:.1%}) already had a Vitamin Product Code.")
print(f"     (The model correctly identified the context despite redactions).")

# --- CHECK B: THE "HIDDEN GEMS" (VALUE ADD) ---
# How many were found under WRONG or GENERIC codes (e.g., 1932 - Other)?
# This demonstrates the true value of the NLP pipeline over simple code filtering.
hidden_gems = vitamins_found[~known_code_mask]

print(f"\n2. Discovery (Value Add):")
print(f"   - {len(hidden_gems)} cases ({len(hidden_gems)/total_vitamins:.1%}) were found under generic/incorrect codes.")

print("\n--- Examples of 'Hidden Gems' (Detected as Vitamin but wrong Product Code) ---")
# Display relevant columns to manually verify quality
pd.set_option('display.max_colwidth', 150)
print(hidden_gems[['Product_1', 'Narrative', 'confidence']].head(10))

# --- CHECK C: HIGH CONFIDENCE FILTER ---
# If we want to be conservative, we can filter for high confidence (>90%)
high_conf_gems = hidden_gems[hidden_gems['confidence'] > 0.90]

print(f"\n--- High Confidence Discoveries (>90%) ---")
print(f"Total: {len(high_conf_gems)}")
print(high_conf_gems[['Product_1', 'Narrative', 'confidence']].head(5))

Total 'Recovered' Vitamins from ambiguous data: 8227

1. Code Confirmation:
   - 0 cases (0.0%) already had a Vitamin Product Code.
     (The model correctly identified the context despite redactions).

2. Discovery (Value Add):
   - 8227 cases (100.0%) were found under generic/incorrect codes.

--- Examples of 'Hidden Gems' (Detected as Vitamin but wrong Product Code) ---
   Product_1  \
3       1930   
13      1928   
17      1930   
22      1930   
25      1112   
62      1932   
63      1932   
71      1807   
76      1930   
78      1930   

                                                                                                                  Narrative  \
3                                 ACCIDENTAL *** INGESTION-@ HOME-CLIMBED UP & TOOK 78 ML OF CHILDREN'S *** @ 160 MG PER ML   
13                     3 YO BF - INGETION - MAY HAVE INGESTED 20 CC'S OF ATUSS WITH CODEINE PRIOR TO ARRI VAL - NO SYMPTOMS   
17                                     CHILD *** INGESTION- DRANK 